In [3]:
# -*- coding: utf-8 -*-
"""
Скрипт для работы с VirusTotal API в Google Colab
С проверкой формата хеша
"""

import requests
import json
import re

# ВАШ API КЛЮЧ
API_KEY = "0fd5e5d4b54c089829f1f244b1a370bc87e2441fa4718f0be1b80cbca95b5eda"

def validate_hash(hash_string):
    """
    Проверка корректности хеша
    """
    hash_string = hash_string.strip().lower()

    # SHA-256: 64 символа, только hex
    if len(hash_string) == 64 and re.match(r'^[a-f0-9]+$', hash_string):
        return hash_string, "SHA-256"

    # SHA-1: 40 символов, только hex
    elif len(hash_string) == 40 and re.match(r'^[a-f0-9]+$', hash_string):
        return hash_string, "SHA-1"

    # MD5: 32 символа, только hex
    elif len(hash_string) == 32 and re.match(r'^[a-f0-9]+$', hash_string):
        return hash_string, "MD5"

    else:
        return None, None

def check_file_virustotal(api_key, file_hash, hash_type):
    """
    Отправка запроса к VirusTotal API
    """
    url = f"https://www.virustotal.com/api/v3/files/{file_hash}"

    headers = {
        "x-apikey": api_key,
        "Accept": "application/json"
    }

    try:
        print(f"\n🔄 Отправка запроса к VirusTotal API...")
        print(f"📋 Тип хеша: {hash_type}")
        print(f"📋 Хеш: {file_hash}")

        response = requests.get(url, headers=headers, timeout=30)

        if response.status_code == 200:
            print("✅ Запрос успешно выполнен!")
            return response.json()
        elif response.status_code == 400:
            print("❌ Ошибка 400: Неверный формат хеша")
            print("   Убедитесь, что хеш корректный (MD5: 32 символа, SHA-1: 40, SHA-256: 64)")
            return None
        else:
            print(f"❌ Ошибка HTTP: {response.status_code}")
            print(f"Сообщение: {response.text}")
            return None

    except Exception as e:
        print(f"❌ Ошибка: {e}")
        return None

def display_results(data):
    """
    Отображение результатов
    """
    if not data:
        return

    print("\n" + "="*70)
    print("📊 РЕЗУЛЬТАТЫ СКАНИРОВАНИЯ")
    print("="*70)

    try:
        attributes = data.get('data', {}).get('attributes', {})

        print(f"\n📁 Файл: {attributes.get('meaningful_name', 'Неизвестно')}")
        print(f"📏 Размер: {attributes.get('size', 0)} байт")
        print(f"📝 Тип: {attributes.get('type_description', 'Неизвестно')}")

        stats = attributes.get('last_analysis_stats', {})
        malicious = stats.get('malicious', 0)

        print(f"\n🛡️ Статистика проверки:")
        print(f"   🔴 Вредоносных: {malicious}")
        print(f"   🟡 Подозрительных: {stats.get('suspicious', 0)}")
        print(f"   🟢 Безопасных: {stats.get('harmless', 0)}")
        print(f"   ⚪ Не обнаружено: {stats.get('undetected', 0)}")

        if malicious > 0:
            print(f"\n⚠️  ФАЙЛ ОПАСЕН! Обнаружен {malicious} антивирусами")
        else:
            print(f"\n✅ Файл безопасен (по данным VirusTotal)")

    except Exception as e:
        print(f"❌ Ошибка при обработке: {e}")

# Основная программа
print("🔐 VT Scanner - Проверка файлов через VirusTotal")
print("="*70)

print(f"\n✅ Используется API ключ")

print("\n📌 ДОСТУПНЫЕ ТЕСТОВЫЕ ХЕШИ:")
print("   1. EICAR тест (SHA-256): 275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f")
print("   2. EICAR тест (MD5): 44d88612fea8a8f36de82e1278abb02f")
print("   3. EICAR тест (SHA-1): 3395856ce81f2b7382dee72602f798b642f14140")
print("   4. Безопасный файл: введите свой хеш")

print("\n🔍 ВВЕДИТЕ ХЕШ ФАЙЛА:")
file_hash = input("👉 Hash: ").strip()

# Валидация хеша
valid_hash, hash_type = validate_hash(file_hash)

if not valid_hash:
    print("\n❌ Некорректный хеш!")
    print("   Требования:")
    print("   • MD5: 32 hex символа")
    print("   • SHA-1: 40 hex символов")
    print("   • SHA-256: 64 hex символа")
    print("\n📋 Используем тестовый хеш EICAR (SHA-256):")
    valid_hash = "275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f"
    hash_type = "SHA-256"
    print(f"   {valid_hash}")

print("\n" + "="*70)
print("🚀 ВЫПОЛНЕНИЕ ЗАПРОСА...")

# Выполняем запрос
result = check_file_virustotal(API_KEY, valid_hash, hash_type)

if result:
    display_results(result)

    # Сохраняем результат
    with open('vt_result.json', 'w') as f:
        json.dump(result, f, indent=2)
    print("\n📄 Результат сохранен в 'vt_result.json'")
else:
    print("\n❌ Не удалось получить результаты")

print("\n" + "="*70)
print("✅ Скрипт завершил работу")

🔐 VT Scanner - Проверка файлов через VirusTotal

✅ Используется API ключ

📌 ДОСТУПНЫЕ ТЕСТОВЫЕ ХЕШИ:
   1. EICAR тест (SHA-256): 275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f
   2. EICAR тест (MD5): 44d88612fea8a8f36de82e1278abb02f
   3. EICAR тест (SHA-1): 3395856ce81f2b7382dee72602f798b642f14140
   4. Безопасный файл: введите свой хеш

🔍 ВВЕДИТЕ ХЕШ ФАЙЛА:
👉 Hash: 275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f

🚀 ВЫПОЛНЕНИЕ ЗАПРОСА...

🔄 Отправка запроса к VirusTotal API...
📋 Тип хеша: SHA-256
📋 Хеш: 275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f
✅ Запрос успешно выполнен!

📊 РЕЗУЛЬТАТЫ СКАНИРОВАНИЯ

📁 Файл: eicar.com
📏 Размер: 68 байт
📝 Тип: Powershell

🛡️ Статистика проверки:
   🔴 Вредоносных: 67
   🟡 Подозрительных: 0
   🟢 Безопасных: 0
   ⚪ Не обнаружено: 2

⚠️  ФАЙЛ ОПАСЕН! Обнаружен 67 антивирусами

📄 Результат сохранен в 'vt_result.json'

✅ Скрипт завершил работу
